# Multimodal (Image) SFT on GEAP — Oral Disease Classification

This notebook is a **thin demo** that calls into the tested `geap_tuning` package;
all the real logic lives in `src/geap_tuning/sft_vision/` and is unit-tested. It
ports https://github.com/jswortz/dental-fine-tune-26 into this repo's shape and is
the repo's **first multimodal (image) example**.

**What's different from text SFT:** each training record's user turn carries an
image as a `fileData` part (a `mimeType` + a `gs://` `fileUri`) alongside the text
prompt; the model turn is the class label. The tuning call itself is *identical*
to text SFT, so this reuses `geap_tuning.sft.tune.launch_sft_job`.

**Dataset:** Kaggle *Multi-Class Oral Disease Detection Dataset*
(`singh868/multi-class-oral-disease-detection-dataset`), by Rahul Singh,
licensed **CC BY-SA 4.0**. Five classes: calculus, cancer, caries, gingivitis,
ulcer.

> **⚠️ Cost & prerequisites.** This launches **two** live tuning jobs and calls a
> tuned endpoint per validation/test image — it **incurs GEAP tuning + inference
> cost**. It also needs: a real `.env` + `gcloud auth`, a Kaggle token in
> `KAGGLE_API_TOKEN`, and the optional `vision` dependency group
> (`uv sync --group vision`, which pulls in `kagglehub`). Keep `PER_CLASS` small.

See `docs/notes/multimodal-sft.md` for the full write-up.

In [ ]:
from geap_tuning.config import genai_client, load_config

cfg = load_config()
client = genai_client(cfg)
cfg

## 1. Download the dataset

`download_dataset()` uses **kagglehub** (optional `vision` group) and your
`KAGGLE_API_TOKEN`. If you already have the dataset locally, skip this cell and
set `source_dir = Path("/path/to/oral-disease")` instead.

In [ ]:
from geap_tuning.sft_vision.data import download_dataset

source_dir = download_dataset()
source_dir

## 2. Downsample + build the multimodal `contents` JSONL

`build_vision_dataset` infers each image's class (filename prefix) and split
(path), takes a **balanced per-class** sample, copies the selection under
`OUT_DIR`, and writes one JSONL manifest per split. Keep `PER_CLASS` small to
bound cost — the reference used 200/40/40.

In [ ]:
from pathlib import Path

from geap_tuning.sft_vision.data import build_vision_dataset

OUT_DIR = Path("datasets/sft_vision_oral")
GCS_PREFIX = "sft_vision_oral"
PER_CLASS = {"train": 50, "val": 10, "test": 10}

dataset = build_vision_dataset(
    source_dir,
    OUT_DIR,
    bucket=cfg.bucket,
    gcs_prefix=GCS_PREFIX,
    per_class=PER_CLASS,
)
{split: len(payload["items"]) for split, payload in dataset.items()}

In [ ]:
# Peek at one built record (image fileData part + prompt, then the label).
dataset["train"]["records"][0]

## 3. Stage images + JSONL to GCS

Every selected image is uploaded to
`{bucket}/{GCS_PREFIX}/data/{split}/{class}/{file}` (mirroring the local layout
so evaluation can map a `gs://` URI back to its local copy), and each split's
JSONL manifest to `{bucket}/{GCS_PREFIX}/{split}.jsonl`.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.sft_vision.data import image_gcs_uri

jsonl_uris = {}
for split, payload in dataset.items():
    for item in payload["items"]:
        dest = image_gcs_uri(cfg.bucket, GCS_PREFIX, item.split, item.class_name, item.filename)
        upload_file(item.local_path, dest)
    jsonl_uris[split] = upload_file(payload["jsonl"], f"{cfg.bucket}/{GCS_PREFIX}/{split}.jsonl")
    print(f"staged {len(payload['items'])} images + JSONL for split={split}")
jsonl_uris

## 4. Launch the hyperparameter sweep

Two configs, each a **separate** SFT job (reused by display name if it already
exists). This is the exact `launch_sft_job` call the text SFT example makes — only
the records are multimodal.

In [ ]:
from geap_tuning.jobs import (
    find_tuning_job_by_display_name,
    tuned_endpoint,
    wait_for_tuning_job,
)
from geap_tuning.sft.tune import launch_sft_job

BASE_MODEL = "gemini-2.5-flash-lite"
EXPERIMENTS = [
    {"name": "baseline", "epochs": 2, "learning_rate_multiplier": 1.0, "adapter_size": 8},
    {"name": "wide", "epochs": 3, "learning_rate_multiplier": 2.0, "adapter_size": 16},
]

endpoints = {}
for exp in EXPERIMENTS:
    display_name = f"geap-sft-vision-{exp['name']}"
    job = find_tuning_job_by_display_name(client, display_name)
    if job is None:
        job = launch_sft_job(
            client,
            train_uri=jsonl_uris["train"],
            val_uri=jsonl_uris["val"],
            display_name=display_name,
            base_model=BASE_MODEL,
            epochs=exp["epochs"],
            adapter_size=exp["adapter_size"],
            learning_rate_multiplier=exp["learning_rate_multiplier"],
            labels=cfg.labels,
        )
        print(f"[{exp['name']}] launched {job.name}")
    else:
        print(f"[{exp['name']}] reusing {job.name} ({job.state})")
    job = wait_for_tuning_job(client, job.name)
    endpoints[exp["name"]] = tuned_endpoint(job)
endpoints

## 5. Evaluate each config on the validation split

`run_image_eval` takes a `predict_fn` so the logic stays testable. Here the
closure loads each image's **local** bytes (mapped back from its `gs://` URI),
sends them to the tuned endpoint, and `parse_prediction` canonicalizes the
free-text answer to one of the five labels.

In [ ]:
from collections.abc import Callable

from google.genai import types

from geap_tuning.inference import generate
from geap_tuning.schemas import Record
from geap_tuning.sft_vision.data import PROMPT
from geap_tuning.sft_vision.evaluate import (
    image_gcs_uri_of,
    resolve_local_path,
    run_image_eval,
)


def make_predict(endpoint: str) -> Callable[[Record], str]:
    def predict(record: Record) -> str:
        mime = record["contents"][0]["parts"][0]["fileData"]["mimeType"]
        local_path = resolve_local_path(image_gcs_uri_of(record), OUT_DIR)
        part = types.Part.from_bytes(data=local_path.read_bytes(), mime_type=mime)
        return generate(client, endpoint, [part, PROMPT])

    return predict


val_results = {}
for name, endpoint in endpoints.items():
    val_results[name] = run_image_eval(dataset["val"]["records"], make_predict(endpoint))
    print(f"[{name}] val accuracy={val_results[name]['accuracy']:.3f}")

## 6. Select the best config, then score it on the test split

In [ ]:
from geap_tuning.sft_vision.evaluate import select_best_experiment

best_name = select_best_experiment(val_results)
test_metrics = run_image_eval(dataset["test"]["records"], make_predict(endpoints[best_name]))

print(f"{'experiment':<12}{'val_acc':>9}{'val_f1':>9}")
for name in sorted(val_results):
    m = val_results[name]
    mark = "  <- best" if name == best_name else ""
    print(f"{name:<12}{m['accuracy']:>9.3f}{m['macro_f1']:>9.3f}{mark}")
print(
    f"\nBest '{best_name}' TEST: "
    f"acc={test_metrics['accuracy']:.3f} f1={test_metrics['macro_f1']:.3f}"
)

## Next steps

- **Text SFT** — `01_sft.ipynb` for the single-modality version of this flow.
- **Preference tuning (DPO)** / **RLFT** — `02`/`03` build on SFT.
- **More data / classes** — raise `PER_CLASS`, or adapt `LABEL_MAP` + `PROMPT` in
  `geap_tuning.sft_vision.data` to your own image-classification task; the rest of
  the pipeline is unchanged.